# 1. LGCA fundamentals and random movement

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/sisyga/biolgca/blob/aidevelop/docs/source/tutorials/01_fundamentals.ipynb)

This lesson builds and runs a first lattice-gas cellular automaton.
We will inspect channel states, run unbiased movement, compare
boundary conditions and verify that a seed reproduces the same
stochastic trajectory.

**Learning objectives**

- relate lattice nodes, velocity channels and propagation;
- build a model with `get_lgca` and choose one of its standard models;
- run it, and record and plot density and population; and
- use boundaries and seeds as explicit experimental choices.

In [ ]:
# In Google Colab this cell installs BioLGCA (about a minute); elsewhere it does nothing.
import importlib.util
import subprocess
import sys

if "google.colab" in sys.modules and importlib.util.find_spec("lgca") is None:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q",
                    "biolgca @ git+https://github.com/sisyga/biolgca@aidevelop"], check=True)

## States, interaction and propagation

A node stores one Boolean value per channel. An occupied velocity
channel represents a cell that will propagate to the corresponding
neighbor. Rest channels, when present, hold cells at the same node.

A time step first applies the stochastic interaction. Random walk
chooses an admissible channel state without a preferred direction.
Deterministic propagation then moves the velocity-channel cells.
Boundary conditions determine what happens when movement reaches
the edge of the simulated domain.


In [ ]:
import matplotlib.pyplot as plt
import numpy as np

from lgca import get_lgca

## A first model

`get_lgca` builds a model from a few choices: the lattice (`geometry`,
`dims`) and its boundary (`bc`), the initial `density` of cells per
channel, the number of rest channels, the interaction and the random
`seed`. The interaction is the name of one of the standard models that
come with BioLGCA; here cells move by a random walk.

In [ ]:
def make_model(bc="periodic", seed=11, density=0.15):
    return get_lgca(
        geometry="square",
        dims=(18, 18),
        bc=bc,
        density=density,
        restchannels=0,
        interaction="random_walk",
        seed=seed,
    )


lgca = make_model()
print("channels per node:", lgca.K)
print("standard models of this lattice:", lgca.interactions)

The standard models cover movement (random walk, alignment,
aggregation, chemotaxis, ...), growth (birth and death, go-or-grow) and
an excitable medium; identity-based models (`ib=True`), whose cells
carry labels and traits, add published research models. Lesson 3 shows that each name stands for a
list of rules, and how to write a model rule by rule so that every part
can be seen, combined and varied.

In [ ]:
lgca.timeevo(timesteps=1, record=True, showprogress=False)

before = lgca.data["nodes"][0]
after = lgca.data["nodes"][1]
print("state shape (x, y, channels):", before.shape)
print("particles before and after:", before.sum(), after.sum())
print("sites whose channel state changed:", np.any(before != after, axis=-1).sum())

`timeevo` runs the model and records what we ask for: the channel
states with `record=True`, the population with `recordN=True`, and the
density of every node by default. `lgca.data` holds the recordings by
name. Random reorientation and propagation change local channel states,
while this particle-conserving interaction leaves the total number
unchanged. The recorded node arrays exclude boundary ghost nodes, so
their first two axes match the requested 18 by 18 domain.

In [ ]:
lgca = make_model()
lgca.timeevo(timesteps=20, record=True, recordN=True, showprogress=False)

fig, axes = plt.subplots(1, 2, figsize=(8, 3.4), constrained_layout=True)
for axis, density_map, title in zip(
    axes,
    (lgca.data["density"][0], lgca.data["density"][-1]),
    ("initial density", "density after 20 steps"),
):
    image = axis.imshow(density_map.T, origin="lower", vmin=0, vmax=4)
    axis.set_title(title)
    axis.set_xlabel("x")
    axis.set_ylabel("y")
fig.colorbar(image, ax=axes, label="particles per node", shrink=0.8)
plt.show()
plt.close(fig)

print("recorded population:", lgca.data["population"])

The recorded densities can also be played as an animation. When
an animation is the last line of a cell, the notebook shows a
player; `save_path="random_walk.gif"` would also write it to a file.


In [ ]:
lgca.animate_density(figsize=(4.5, 4))

## Boundary conditions are model assumptions

Periodic boundaries connect opposite edges. Reflecting boundaries
represent a no-flux wall. We change only the boundary and seed the
two runs identically, which makes the comparison controlled.


In [ ]:
boundary_models = {}
for bc in ("periodic", "reflecting"):
    model = make_model(bc=bc, seed=17)
    model.timeevo(timesteps=20, showprogress=False)
    boundary_models[bc] = model

fig, axes = plt.subplots(1, 2, figsize=(8, 3.4), constrained_layout=True)
for axis, (bc, model) in zip(axes, boundary_models.items()):
    axis.imshow(model.data["density"][-1].T, origin="lower", vmin=0, vmax=4)
    axis.set_title(bc)
    axis.set_xlabel("x")
    axis.set_ylabel("y")
plt.show()
plt.close(fig)

With a spatially uniform random initialization, a short run may not
make edge effects dramatic. A useful follow-up experiment is to
initialize cells near one edge and compare escape or accumulation.


## Reproducibility means rerunning the same stochastic history

A seed does not remove stochasticity. It selects one reproducible
realization, which is useful for debugging and paired comparisons.
Scientific uncertainty still requires multiple seeds, introduced
in lesson 3.

In [ ]:
def trajectory(seed):
    model = make_model(seed=seed)
    model.timeevo(timesteps=20, record=True, showprogress=False)
    return model.data["nodes"]


first, second, different = trajectory(42), trajectory(42), trajectory(43)
assert np.array_equal(first, second)
print("same seed gives identical trajectory:", np.array_equal(first, second))
print("different seed gives identical trajectory:", np.array_equal(first, different))

## Interpretation

This model represents unbiased motion with excluded channel
occupancy. It is a baseline, not a biological explanation for
directed migration or collective order. Later lessons will add
specific directional mechanisms.

## Exercises

1. Change `density` to 0.05 and 0.5. How does crowding affect the
   final density map?
2. Add one rest channel (`restchannels=1`). Does the spatial spread
   change over the same 20 steps?
3. Start with a compact patch of cells near the left boundary: build
   an array of channel states and pass it as `nodes=` to `get_lgca`
   instead of `density`. Compare periodic, reflecting and absorbing
   boundaries.
4. Write down the seed and every parameter before sharing a figure.